In [ ]:
transformer_analysis = spark.sql("""
with hourly as (
    select 
        m.transformer_id,
        date_trunc('hour', r.timestamp) as hour_ts,
        round(sum(r.kwh), 2) as current_load
    from `electric_raw_stage`.`transformer_meter_mapping` m
    join `electric_raw_stage`.`meter_usage` r on m.meter_id = r.meter_id
    group by m.transformer_id, date_trunc('hour', r.timestamp)
),
lagged as (
    select 
        *,
        lag(current_load) over (partition by transformer_id order by hour_ts) as previous_load
    from hourly
)
select 
    transformer_id,
    hour_ts,
    current_load,
    round(previous_load, 2) as previous_load,
    round(abs((current_load - previous_load) / previous_load) * 100, 2) as percent_change,
    case 
        when current_load > previous_load then 'Increase'
        when current_load < previous_load then 'Decrease'
        else 'no change'
    end as load_change
from lagged
order by transformer_id, hour_ts desc
""")

spark.sql("create database if not exists `electric_analysis_dev`")

transformer_analysis.coalesce(1).write \
    .mode("overwrite") \
    .format("parquet") \
    .option("path", "s3://ops-autopilot-data/transformed/transformer_load_analysis/") \
    .saveAsTable("`electric_analysis_dev`.`transformer_hourly_analysis`")